In [1]:
# ============================================
# Imports
# ============================================

from pathlib import Path
from collections import Counter, defaultdict
import gzip
import re
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# ============================================
# Configuration
# ============================================

# Workflow repository
REPO = Path.home() / "Code" / "processing" / "riboseq"

# Project data directory
PROJECT = Path("/mnt/d/Ibnu/riboseq/AT")

# nf-core output currently used for validation/post-processing
NFCORE_RUN = PROJECT / "nfcore" / "10pct"

# Full biological annotation
GFF3 = REPO / "refs" / "at.gff3.gz"

# Output table directory
TABLES = PROJECT / "TABLES"
TABLES.mkdir(parents=True, exist_ok=True)

# Output file
ANNOTATION_OUT = TABLES / "annotation.10pct.tsv"

In [3]:
# ============================================
# Input check
# ============================================

paths = {
    "REPO": REPO,
    "PROJECT": PROJECT,
    "NFCORE_RUN": NFCORE_RUN,
    "GFF3": GFF3,
    "TABLES": TABLES,
}

for name, path in paths.items():
    print(f"{name:12s}: {path}")
    print(f"{'exists':12s}: {path.exists()}")
    print()

REPO        : /home/ha-ibnu/Code/processing/riboseq
exists      : True

PROJECT     : /mnt/d/Ibnu/riboseq/AT
exists      : True

NFCORE_RUN  : /mnt/d/Ibnu/riboseq/AT/nfcore/10pct
exists      : True

GFF3        : /home/ha-ibnu/Code/processing/riboseq/refs/at.gff3.gz
exists      : True

TABLES      : /mnt/d/Ibnu/riboseq/AT/TABLES
exists      : True



In [20]:
# ============================================
# Build annotation table from GFF3
# ============================================

def parse_gff3_attributes(attr: str) -> dict:
    out = {}
    for item in attr.strip().split(";"):
        if item and "=" in item:
            k, v = item.split("=", 1)
            out[k] = v
    return out


records = []

with gzip.open(GFF3, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue

        chrom, source, feature, start, end, score, strand, phase, attr = line.rstrip().split("\t")
        attrs = parse_gff3_attributes(attr)

        if feature not in ["mRNA", "exon", "CDS", "five_prime_UTR", "three_prime_UTR"]:
            continue

        records.append({
            "Chr": chrom,
            "Feature": feature,
            "Start": int(start),
            "End": int(end),
            "Strand": strand,
            "ID": attrs.get("ID"),
            "Parent": attrs.get("Parent"),
            "Name": attrs.get("Name"),
            "Length": int(end) - int(start) + 1,
        })

gff = pd.DataFrame(records)

transcripts = (
    gff[gff["Feature"] == "mRNA"]
    .rename(columns={
        "ID": "Transcript_ID",
        "Parent": "Gene_ID",
        "Start": "Transcript_start",
        "End": "Transcript_end",
    })
    [[
        "Transcript_ID",
        "Gene_ID",
        "Chr",
        "Strand",
        "Transcript_start",
        "Transcript_end",
    ]]
    .copy()
)

def sum_feature(feature, colname):
    return (
        gff[gff["Feature"] == feature]
        .groupby("Parent", as_index=False)["Length"]
        .sum()
        .rename(columns={
            "Parent": "Transcript_ID",
            "Length": colname,
        })
    )

annotation = (
    transcripts
    .merge(sum_feature("exon", "Transcript_length"), on="Transcript_ID", how="left")
    .merge(sum_feature("five_prime_UTR", "UTR5_length"), on="Transcript_ID", how="left")
    .merge(sum_feature("CDS", "CDS_length"), on="Transcript_ID", how="left")
    .merge(sum_feature("three_prime_UTR", "UTR3_length"), on="Transcript_ID", how="left")
)

for col in ["Transcript_length", "UTR5_length", "CDS_length", "UTR3_length"]:
    annotation[col] = annotation[col].fillna(0).astype(int)

annotation["UTR_length"] = annotation["UTR5_length"] + annotation["UTR3_length"]

annotation.head()

,Transcript_ID,Gene_ID,Chr,Strand,Transcript_start,Transcript_end,Transcript_length,UTR5_length,CDS_length,UTR3_length,UTR_length
0,AT1G01010.1,AT1G01010,Chr1,+,3631,5899,1688,129,1290,269,398
1,AT1G01020.2,AT1G01020,Chr1,-,6788,8737,167,71,576,0,71
2,AT1G01020.6,AT1G01020,Chr1,-,6788,8737,144,144,315,0,144
3,AT1G01020.1,AT1G01020,Chr1,-,6788,9130,0,464,738,0,464
4,AT1G01020.4,AT1G01020,Chr1,-,6788,9130,0,0,711,0,0


In [21]:
# ============================================
# Save annotation table
# ============================================

ANNOTATION_OUT = TABLES / "annotation.10pct.tsv"

annotation.to_csv(ANNOTATION_OUT, sep="\t", index=False)

print(ANNOTATION_OUT)
print(annotation.shape)

/mnt/d/Ibnu/riboseq/AT/TABLES/annotation.10pct.tsv
(52270, 11)
